In [ ]:
import typing
import utils
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt

In [ ]:
basedir = "../5_task_transfer"

In [ ]:
df = utils.subset_data(
    basedir=basedir,
    index={
        # "model.from_pretrained": lambda x: x is not None,
        "dataset.concurrent_reg": lambda x: x == 5,
    },
)
df.sample(5)

In [ ]:
td_prob_info = utils.extract_sweep_properties(
    utils.gather_sweeps(basedir), ["dataset.td_prob"]
)
[*td_prob_info.items()][0]

In [ ]:
from typing import Literal


def get_pretraining_condition(row) -> Literal["REF-BACK", "N-BACK", "NA"]:
    this_pretraining = row["from_pretrained"]
    if isinstance(this_pretraining, str):
        pretraining_sweep_id = this_pretraining.split("/")[1]
        return (
            "REF-BACK"
            if td_prob_info[pretraining_sweep_id]["dataset.td_prob"] == 0
            else "N-BACK"
        )
    return "NA"


# map td_prob_info to `df` as a new column called 'pretraining_condition'
df["pretraining_condition"] = df.apply(
    get_pretraining_condition,
    axis=1,
)

df.loc[df["from_pretrained"] == None]["pretraining_condition"] = "NA"

In [ ]:
sns.lineplot(
    df, x="epoch", y="eval_acc", hue="pretraining_condition", style="model_class"
)

In [ ]:
def shuffled(collection: typing.Collection):
    import random

    return random.sample(collection, len(collection))
    # Define a mapping of model_class to colors


g = sns.FacetGrid(
    df,
    col="model_class",
    row="td_prob",
    hue="pretraining_condition",
    sharex=True,
    sharey=True,
    palette=sns.color_palette(["#1b9e77", "#d95f02", "#7570b3"]),
)
g.map_dataframe(
    sns.lineplot,
    x="epoch",
    y="test_acc",
    alpha=1,
    errorbar="se",
)

alpha = 0.15
background_colors = dict(
    zip(
        df.model_class.unique(),
        zip(
            shuffled(["#fc8d59", "#ffffbf", "#91bfdb"]),
            [alpha] * 10,
        ),
    )
)

# Apply background color to each subplot based on model_class


g.add_legend()
for ax in g.axes.flat:
    ax.set_ylim(0.3, 1)
    ax.set_xlim(0, 40)
    ax.grid()

    model_class = ax.get_title().split(" | ")[1].split(" = ")[1]
    if model_class in background_colors:
        ax.set_facecolor(background_colors[model_class])

plt.show()